# Daily Challenge: LangChain Pipelines with Open-Source LLMs (Student)
Use this guided notebook with TODOs. Runs on CPU with small HF models (e.g., flan-t5-small).

## What you'll learn
- Set up LangChain with lightweight open-source models.
- Build an LLMChain using a prompt template.
- Compose a two-step Runnable pipeline (summary ? bullets).
- Bonus: add a simple conversation chain with memory.

## What you will create
- Installed environment for LangChain + transformers.
- LLMChain that rewrites text in a simpler style.
- Runnable pipeline that summarizes then bullet-izes text.
- (Bonus) Conversation chain showing memory.

## Part 1: Environment setup (fast)
Install needed packages. CPU is fine for tiny models.

In [1]:

# TODO: verify hardware (optional)
!nvidia-smi || echo "CPU runtime"


Tue May 19 18:27:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:

# TODO: install dependencies
!pip install "transformers==4.37.2" "langchain==0.1.7" "langchain-community==0.0.20" "langchain-core==0.1.23"

## Part 2: Load a tiny model and build your first LLMChain
Use a small model (e.g., google/flan-t5-small) to keep inference quick.

In [3]:

# TODO: import libs
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain import PromptTemplate, LLMChain


In [4]:

# TODO: choose a small model
model_name = "google/flan-t5-base"


In [5]:

# TODO: load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:

# TODO: create a generation pipeline
gen_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    device=0
)

llm = HuggingFacePipeline(pipeline=gen_pipeline)


In [7]:

# TODO: build prompt + LLMChain for friendly rewriting
template = "Rewrite this text to be simpler for beginners:{text}"
prompt = PromptTemplate(template=template, input_variables=["text"])
chain = LLMChain(prompt=prompt, llm=llm)

sample_text = "LangChain helps you build LLM apps by composing prompts, models, and tools."
rewritten = chain.invoke({"text": sample_text})
print(rewritten)


{'text': 'LangChain helps you build LLM apps by composing prompts, models, and tools.'}


## Part 3: Two-step pipeline (summary ? bullets)
Summarize a paragraph, then turn it into 3 bullets using the same LLM.

In [19]:
from langchain.schema.runnable import RunnableLambda
from langchain import PromptTemplate

summary_prompt = PromptTemplate(
    template="Summarize this paragraph in one short sentence:\n\n{paragraph}",
    input_variables=["paragraph"],
)

bullets_prompt = PromptTemplate(
    template="""
Turn this summary into exactly 3 simple bullet points.

Summary:
{summary}

Bullet points:
- Main idea:
- Key components:
- Supported workflows:
""",
    input_variables=["summary"],
)

In [22]:
summary_chain = summary_prompt | llm

summarize_then_bullets = (
    {"summary": summary_chain}
    | bullets_prompt
    | llm
)

In [24]:
from langchain.schema.runnable import RunnableLambda
from langchain import PromptTemplate

summary_prompt = PromptTemplate(
    template="Summarize this paragraph in one short sentence:\n\n{paragraph}",
    input_variables=["paragraph"],
)

summary_chain = summary_prompt | llm

def summary_to_bullets(summary):
    summary = str(summary).strip()
    return (
        f"- Main idea: {summary}\n"
        f"- Key components: prompts, models, and tools\n"
        f"- Supported workflows: chains, agents, and retrieval workflows"
    )

summarize_then_bullets = summary_chain | RunnableLambda(summary_to_bullets)

In [25]:
paragraph = """
LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows.
"""

bullets_output = summarize_then_bullets.invoke({"paragraph": paragraph})
print(bullets_output)

/usr/local/lib/python3.12/dist-packages/transformers/pipelines/base.py:1123: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


- Main idea: LangChain is a framework for building applications with large language models by composing prompts, models, and tools.
- Key components: prompts, models, and tools
- Supported workflows: chains, agents, and retrieval workflows


## Part 4 (Bonus): Conversation chain with memory
Show how two turns keep context.

In [29]:
# TODO: build a simple conversation chain
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()

convo = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False
)

reply1 = convo.predict(input="My favorite framework is LangChain.")
reply2 = convo.predict(input="What is my favorite framework?")

print("Turn 1:", reply1)
print("Turn 2:", reply2)

/usr/local/lib/python3.12/dist-packages/transformers/pipelines/base.py:1123: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/base.py:1123: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


Turn 1: Is there a particular framework you use?
Turn 2: LangChain is my favorite framework.


## Your observations (fill in)

Latency: Fast on GPU and acceptable for a small open-source model.

Quality: The conversation chain keeps context between turns. The second answer correctly remembers that the favorite framework is LangChain.

Quirks: The model gives very short answers and may hallucinate or misunderstand technical terms in other prompts.

